<a href="https://colab.research.google.com/github/flango2023/Pulso_F2/blob/main/fase2_parte2_classificador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pulso_Fase_2: Classificador de Risco com TF-IDF e Regressão Logística

**Projeto:** Pulso: Inteligência Cardiológica com Consciência de Dados  
**Autor:** Richard Schmitz | RM 567951  


---

## Objetivo

Este notebook implementa a Parte 2 da Fase 2 do projeto Pulso.  
A partir de frases médicas rotuladas, o sistema:

1. Carrega o dataset `frases_risco.csv` com frases e rótulos de risco
2. Transforma as frases em vetores numéricos com TF-IDF
3. Treina um classificador de Regressão Logística
4. Avalia o desempenho geral do modelo
5. Analisa se o modelo performa de forma diferente para frases com perfil feminino atípico vs. masculino típico

Essa última etapa é a contribuição original do projeto Pulso:  
verificar se o viés de gênero documentado na Fase 1 se manifesta também no classificador.

## 1. Configuração do ambiente

> **Google Colab:** faça upload de `frases_risco.csv` antes de executar.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Detecta ambiente de execução
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False

# Define caminho do dataset de acordo com o ambiente
CAMINHO_CSV = '/content/frases_risco.csv' if COLAB else '../data/frases_risco.csv'

print(f'Ambiente: {"Google Colab" if COLAB else "Local"}')
print('Bibliotecas carregadas com sucesso.')

Ambiente: Google Colab
Bibliotecas carregadas com sucesso.


## 2. Carregamento e inspeção do dataset

In [2]:
df = pd.read_csv(CAMINHO_CSV)

print(f'Total de registros: {len(df)}')
print(f'\nDistribuição por nível de risco:')
print(df['nivel_risco'].value_counts())
print(f'\nDistribuição por perfil de gênero:')
print(df['perfil_genero'].value_counts())
print()
df.head(10)

Total de registros: 30

Distribuição por nível de risco:
nivel_risco
alto risco     15
baixo risco    15
Name: count, dtype: int64

Distribuição por perfil de gênero:
perfil_genero
masculino típico    13
feminino atípico    11
ambos                6
Name: count, dtype: int64



,frase,nivel_risco,perfil_genero
0,sinto dor forte no peito que irradia para o br...,alto risco,masculino típico
1,pressão intensa no tórax que dura minutos e pa...,alto risco,masculino típico
2,falta de ar durante a noite e coração acelerad...,alto risco,masculino típico
3,tontura e suor frio ao subir escadas quase des...,alto risco,masculino típico
4,coração irregular como se pulasse uma batida c...,alto risco,masculino típico
5,cansaço extremo há duas semanas sem conseguir ...,alto risco,feminino atípico
6,náusea constante e dor estranha no maxilar e n...,alto risco,feminino atípico
7,falta de ar ao varrer a casa e tornozelos inch...,alto risco,feminino atípico
8,fadiga que não passa e sensação de peso no est...,alto risco,feminino atípico
9,desconforto no peito que não é dor mas impede ...,alto risco,feminino atípico


## 3. Pré-processamento com TF-IDF

O TF-IDF (Term Frequency–Inverse Document Frequency) transforma cada frase  
em um vetor numérico onde cada posição representa uma palavra do vocabulário.  
Palavras frequentes em um documento mas raras no corpus recebem peso maior,  
capturando termos mais discriminativos para a classificação.

In [3]:
# Separa features (frases) e rótulos (nivel_risco)
# perfil_genero NÃO entra no treino — é usado apenas na análise de viés
X = df['frase']
y = df['nivel_risco']

# Divisão treino/teste estratificada para manter proporção de classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# TF-IDF com bigramas: captura expressões como "dor no" e "falta de"
# min_df=1 garante que todos os termos sejam incluídos (dataset pequeno)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train_tfidf = vectorizer.fit_transform(X_train)  # aprende vocabulário e transforma
X_test_tfidf  = vectorizer.transform(X_test)        # apenas transforma, sem reaprender

print(f'Vocabulário TF-IDF: {len(vectorizer.vocabulary_)} termos')
print(f'Amostras de treino : {X_train_tfidf.shape[0]}')
print(f'Amostras de teste  : {X_test_tfidf.shape[0]}')

Vocabulário TF-IDF: 251 termos
Amostras de treino : 21
Amostras de teste  : 9


## 4. Treinamento do classificador

In [4]:
# Regressão Logística: interpreta os pesos TF-IDF como probabilidade de cada classe
# random_state=42 garante reprodutibilidade
# max_iter=1000 evita warning de convergência com vocabulário grande
modelo = LogisticRegression(random_state=42, max_iter=1000)
modelo.fit(X_train_tfidf, y_train)

# Exibe a ordem das classes — importante para interpretar os coeficientes corretamente
print(f'Classes do modelo: {modelo.classes_}')
print('Modelo treinado com sucesso.')

Classes do modelo: ['alto risco' 'baixo risco']
Modelo treinado com sucesso.


## 5. Avaliação geral do modelo

In [5]:
y_pred = modelo.predict(X_test_tfidf)

acuracia = accuracy_score(y_test, y_pred)
print(f'Acurácia geral: {acuracia:.2%}\n')

print('Relatório de classificação:')
print(classification_report(y_test, y_pred))

print('Matriz de confusão:')
cm = confusion_matrix(y_test, y_pred, labels=['alto risco', 'baixo risco'])
cm_df = pd.DataFrame(
    cm,
    index=['Real: alto risco', 'Real: baixo risco'],
    columns=['Previsto: alto risco', 'Previsto: baixo risco']
)
print(cm_df)

Acurácia geral: 66.67%

Relatório de classificação:
              precision    recall  f1-score   support

  alto risco       1.00      0.40      0.57         5
 baixo risco       0.57      1.00      0.73         4

    accuracy                           0.67         9
   macro avg       0.79      0.70      0.65         9
weighted avg       0.81      0.67      0.64         9

Matriz de confusão:
                   Previsto: alto risco  Previsto: baixo risco
Real: alto risco                      2                      3
Real: baixo risco                     0                      4


## 6. Análise de viés de gênero no classificador

Esta é a contribuição original do projeto Pulso nesta fase.  

A hipótese, baseada no eixo temático da Fase 1, é que o modelo pode classificar  
frases com sintomas atípicos (perfil feminino) com menor precisão do que frases  
com sintomas típicos (perfil masculino), porque o vocabulário de alto risco no  
dataset é dominado por termos masculinos clássicos como "dor no peito" e "aperto no tórax".

In [6]:
# Aplica o modelo a todo o dataset para análise por subgrupo
# (não apenas ao conjunto de teste)
X_all_tfidf = vectorizer.transform(df['frase'])
df = df.copy()  # evita SettingWithCopyWarning
df['predicao'] = modelo.predict(X_all_tfidf)
df['correto']  = (df['predicao'] == df['nivel_risco']).astype(int)

# Acurácia por perfil de gênero
analise_vies = df.groupby('perfil_genero').agg(
    total=('correto', 'count'),
    acertos=('correto', 'sum')
).reset_index()
analise_vies['acuracia_%'] = (
    analise_vies['acertos'] / analise_vies['total'] * 100
).round(1)

print('Acurácia do modelo por perfil de gênero:\n')
print(analise_vies.to_string(index=False))

# Falsos negativos em alto risco por perfil
# Falso negativo = paciente de alto risco classificado como baixo risco
alto_risco = df[df['nivel_risco'] == 'alto risco'].copy()
fn_analise = alto_risco.groupby('perfil_genero').agg(
    total_alto_risco=('correto', 'count'),
    classificados_corretamente=('correto', 'sum')
).reset_index()
fn_analise['falsos_negativos'] = (
    fn_analise['total_alto_risco'] - fn_analise['classificados_corretamente']
)
fn_analise['taxa_fn_%'] = (
    fn_analise['falsos_negativos'] / fn_analise['total_alto_risco'] * 100
).round(1)

print('\nFalsos negativos em casos de alto risco por perfil de gênero:')
print('(Falso negativo = paciente de alto risco classificado como baixo risco)\n')
print(fn_analise.to_string(index=False))

print('\nInterpretação:')
print('Uma taxa de falso negativo maior no perfil feminino atípico indica que')
print('o modelo subestima o risco de pacientes com sintomas não clássicos.')
print('Esse padrão reproduz, em escala reduzida, o viés documentado na literatura clínica.')

Acurácia do modelo por perfil de gênero:

   perfil_genero  total  acertos  acuracia_%
           ambos      6        6       100.0
feminino atípico     11        9        81.8
masculino típico     13       12        92.3

Falsos negativos em casos de alto risco por perfil de gênero:
(Falso negativo = paciente de alto risco classificado como baixo risco)

   perfil_genero  total_alto_risco  classificados_corretamente  falsos_negativos  taxa_fn_%
           ambos                 2                           2                 0        0.0
feminino atípico                 6                           4                 2       33.3
masculino típico                 7                           6                 1       14.3

Interpretação:
Uma taxa de falso negativo maior no perfil feminino atípico indica que
o modelo subestima o risco de pacientes com sintomas não clássicos.
Esse padrão reproduz, em escala reduzida, o viés documentado na literatura clínica.


## 7. Teste com novas frases

Demonstração do modelo com frases não vistas durante o treinamento.

In [7]:
novas_frases = [
    # Sintoma típico masculino — esperado: alto risco
    "sinto dor forte no peito e suor frio há uma hora",
    # Sintoma atípico feminino — esperado: alto risco, mas pode ser subclassificado
    "estou com cansaço que não passa e uma sensação estranha no estômago",
    # Baixo risco claro
    "leve dor de cabeça após dia estressante que passou com repouso",
    # Sintoma atípico feminino grave
    "tenho sentido náusea e dor no maxilar que aparece e some sem motivo"
]

novas_tfidf  = vectorizer.transform(novas_frases)
predicoes    = modelo.predict(novas_tfidf)
probabilidades = modelo.predict_proba(novas_tfidf)

# Identifica o índice da classe 'alto risco' para extrair a probabilidade correta
idx_alto = list(modelo.classes_).index('alto risco')

print('Classificação de novas frases:\n')
for frase, pred, prob in zip(novas_frases, predicoes, probabilidades):
    confianca = prob[idx_alto]
    print(f'Frase     : {frase}')
    print(f'Resultado : {pred} (confiança alto risco: {confianca:.1%})')
    print()

Classificação de novas frases:

Frase     : sinto dor forte no peito e suor frio há uma hora
Resultado : alto risco (confiança alto risco: 58.9%)

Frase     : estou com cansaço que não passa e uma sensação estranha no estômago
Resultado : alto risco (confiança alto risco: 55.0%)

Frase     : leve dor de cabeça após dia estressante que passou com repouso
Resultado : baixo risco (confiança alto risco: 41.2%)

Frase     : tenho sentido náusea e dor no maxilar que aparece e some sem motivo
Resultado : alto risco (confiança alto risco: 57.4%)



## 8. Termos mais relevantes por classe

Os coeficientes da Regressão Logística revelam quais termos o modelo  
associa mais fortemente a cada nível de risco.

In [8]:
feature_names = vectorizer.get_feature_names_out()
coeficientes  = modelo.coef_[0]  # vetor de coeficientes para classificação binária

# Para Regressão Logística binária:
# coef_ > 0 → associado à classe positiva (classes_[1])
# coef_ < 0 → associado à classe negativa (classes_[0])
classe_positiva = modelo.classes_[1]
classe_negativa = modelo.classes_[0]

top_positivo = np.argsort(coeficientes)[-10:][::-1]   # maiores coeficientes
top_negativo = np.argsort(coeficientes)[:10]           # menores coeficientes

print(f'Termos mais associados a "{classe_positiva}":')
for idx in top_positivo:
    print(f'  {feature_names[idx]:<40} coef: {coeficientes[idx]:.3f}')

print(f'\nTermos mais associados a "{classe_negativa}":')
for idx in top_negativo:
    print(f'  {feature_names[idx]:<40} coef: {coeficientes[idx]:.3f}')

print('\nObservação:')
print('Se os termos de alto risco forem dominados por sintomas masculinos clássicos')
print('("dor no peito", "aperto", "irradiação"), isso confirma o viés de representação')
print('no vocabulário do modelo — mesmo com um dataset balanceado por gênero.')

Termos mais associados a "baixo risco":
  leve                                     coef: 0.464
  após                                     coef: 0.433
  que passa                                coef: 0.293
  muito                                    coef: 0.180
  com repouso                              coef: 0.175
  ficar                                    coef: 0.173
  após ficar                               coef: 0.173
  leve tontura                             coef: 0.165
  tontura                                  coef: 0.165
  ao                                       coef: 0.162

Termos mais associados a "alto risco":
  dor                                      coef: -0.212
  não                                      coef: -0.201
  que não                                  coef: -0.201
  intensa                                  coef: -0.198
  peito que                                coef: -0.186
  minutos                                  coef: -0.184
  dura                            

---

## Considerações finais

Este classificador demonstra que é possível construir um sistema básico de triagem  
de risco cardiovascular com ferramentas acessíveis. As limitações são documentadas  
intencionalmente como parte da análise de governança:

- Com 30 frases de treinamento, o modelo é funcional mas não generalizável
- A análise de viés por gênero mostra que sintomas atípicos femininos podem ser subclassificados
- Nas fases seguintes, o dataset UCI Cleveland (303 pacientes) substituirá esse corpus sintético

**Este notebook não produz diagnósticos médicos. Tem finalidade exclusivamente acadêmica.**